In [1]:
import math
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.metrics import r2_score


In [2]:
# =====================================================================
# Overnight attention-only training: 6L, 5L, 7L
# Trained on lags 1-50, evaluated on held-out high lags 52,55,58
# =====================================================================

from torch.optim import AdamW
import os

T, rho = 200, 0.9
train_lags = list(range(1, 51))
held_high  = [52, 55, 58]

batch_size = 64
num_steps = 300000        # overnight; reduce to 150000 if needed
lr = 1e-3
weight_decay = 0.1
burn_in = 30
eval_every = 10000

depths_to_train = [6, 5, 7]

x_tr, _, lags_tr = make_dataset_lagset(3000, T, rho, train_lags, seed=1)
x_hi, _, lags_hi = make_dataset_lagset(3000, T, rho, held_high,  seed=3)

x_tr, lags_tr = x_tr.to(device), lags_tr.to(device)
x_hi, lags_hi = x_hi.to(device), lags_hi.to(device)

def masked_loss(pred, y, lags, burn_in):
    Tn  = pred.shape[1]
    pos = torch.arange(Tn, device=pred.device).unsqueeze(0)
    valid = pos >= (lags.unsqueeze(1) + burn_in)
    se = (pred - y) ** 2
    return ((se * valid).sum(1) / valid.sum(1).clamp(min=1)).mean()

m = lambda d: float(np.mean(list(d.values())))

for n_lay in depths_to_train:
    print(f"\n{'='*60}")
    print(f"TRAINING {n_lay}-LAYER ATTENTION-ONLY MODEL")
    print(f"{'='*60}")

    mdl = AutocorrRoPE(
        d_model=64,
        d_head=64,
        n_layers=n_lay,
        n_heads=1,
        use_mlp=False
    ).to(device)

    latest_path = f"sweep_d64_{n_lay}L_attn_lag1_50_latest.pt"
    best_path   = f"sweep_d64_{n_lay}L_attn_lag1_50_best.pt"
    final_path  = f"sweep_d64_{n_lay}L_attn_lag1_50_final.pt"

    if os.path.exists(latest_path):
        mdl.load_state_dict(torch.load(latest_path, map_location=device))
        print(f"  loaded existing latest checkpoint -> {latest_path}")

    opt = AdamW(mdl.parameters(), lr=lr, weight_decay=weight_decay)

    best_ext = -999

    if os.path.exists(best_path):
        print(f"  found existing best checkpoint -> {best_path}")
        tmp = AutocorrRoPE(
            d_model=64,
            d_head=64,
            n_layers=n_lay,
            n_heads=1,
            use_mlp=False
        ).to(device)
        tmp.load_state_dict(torch.load(best_path, map_location=device))
        tmp.eval()
        best_ext = m(correlation_by_lag(tmp, x_hi, lags_hi, device, verbose=False))
        del tmp
        torch.cuda.empty_cache()
        print(f"  starting best_ext = {best_ext:.3f}")

    for step in range(1, num_steps + 1):
        x, y, lags = make_dataset_lagset(batch_size, T, rho, train_lags, seed=None)
        x, y, lags = x.to(device), y.to(device), lags.to(device)

        pred, *_ = mdl(x)
        loss = masked_loss(pred, y, lags, burn_in)

        opt.zero_grad()
        loss.backward()
        opt.step()

        if step % eval_every == 0:
            tr  = m(correlation_by_lag(mdl, x_tr, lags_tr, device, verbose=False))
            ext = m(correlation_by_lag(mdl, x_hi, lags_hi, device, verbose=False))

            print(
                f"  [{n_lay}L-attn] step {step:6d}  "
                f"loss {loss.item():.4f}  train {tr:.3f}  extrap {ext:.3f}"
            )

            torch.save(mdl.state_dict(), latest_path)

            if ext > best_ext:
                best_ext = ext
                torch.save(mdl.state_dict(), best_path)
                print(f"      saved best so far -> {best_path}, extrap={best_ext:.3f}")

    torch.save(mdl.state_dict(), final_path)

    tr  = m(correlation_by_lag(mdl, x_tr, lags_tr, device, verbose=False))
    ext = m(correlation_by_lag(mdl, x_hi, lags_hi, device, verbose=False))

    print(f"  saved final -> {final_path}")
    print(f"  FINAL train {tr:.3f}  extrap {ext:.3f}")

    del mdl
    torch.cuda.empty_cache()

print("\n=== overnight attention-only training complete ===")

NameError: name 'make_dataset_lagset' is not defined